In [3]:
import pandas as pd
import os

# Change this to your project root
BASE = "/Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data"

# Define all files with metadata
FILES = {
    # --- Pre-AI: Guardian ---
    "guardian_cloudflare_pre": {
        "path": f"{BASE}/pre_AI/guardian_cloudflare_outage_2ALL.csv",
        "source": "guardian", "event": "cloudflare_2019", "era": "pre_ai",
    },
    "guardian_facebook_2021": {
        "path": f"{BASE}/pre_AI/guardian_facebook_outage_2021-10.csv",
        "source": "guardian", "event": "facebook_2021", "era": "pre_ai",
    },
    # --- Pre-AI: NYT ---
    "nyt_cloudflare_2019": {
        "path": f"{BASE}/pre_AI/nyt_cloudflare_2019-07-02.csv",
        "source": "nyt", "event": "cloudflare_2019", "era": "pre_ai",
    },
    "nyt_facebook_2021_event": {
        "path": f"{BASE}/pre_AI/nyt_facebook_outage_2021-10-04.csv",
        "source": "nyt", "event": "facebook_2021", "era": "pre_ai",
    },
    "nyt_facebook_2021_month": {
        "path": f"{BASE}/pre_AI/nyt_facebook_outage_2021-10.csv",
        "source": "nyt", "event": "facebook_2021", "era": "pre_ai",
    },
    # --- Post-AI: Guardian ---
    "guardian_cloudflare_2025": {
        "path": f"{BASE}/post_AI/guardian_cloudflare_outage_2025-11-17_2025-1-20.csv",
        "source": "guardian", "event": "cloudflare_2025", "era": "post_ai",
    },
    "guardian_crowdstrike_2024": {
        "path": f"{BASE}/post_AI/guardian_crowdstrike_2024.csv",
        "source": "guardian", "event": "crowdstrike_2024", "era": "post_ai",
    },
    "guardian_aws_2025": {
        "path": f"{BASE}/post_AI/guardian_aws_2025.csv",
        "source": "guardian", "event": "aws_2025", "era": "post_ai",
    },
    # --- Post-AI: NYT ---
    "nyt_cloudflare_2025": {
        "path": f"{BASE}/post_AI/nyt_cloudflare_2025-11-18.csv",
        "source": "nyt", "event": "cloudflare_2025", "era": "post_ai",
    },
    "nyt_crowdstrike_2024": {
        "path": f"{BASE}/post_AI/nyt_crowdstrike_2024.csv",
        "source": "nyt", "event": "crowdstrike_2024", "era": "post_ai",
    },
    "nyt_aws_2025": {
        "path": f"{BASE}/post_AI/nyt_aws_2025.csv",
        "source": "nyt", "event": "aws_2025", "era": "post_ai",
    },
}

print(f"Total files to process: {len(FILES)}")

Total files to process: 11


In [4]:
def load_guardian(path):
    """Load Guardian CSV and standardize columns."""
    df = pd.read_csv(path)
    return df.rename(columns={
        "pub_date": "pub_date",
        "title": "title",
        "section": "section",
        "web_url": "web_url",
        "trailText": "trail_text",
        "bodyText": "body_text",
    })[["pub_date", "title", "section", "web_url", "trail_text", "body_text"]]


def load_nyt(path):
    """Load NYT CSV and standardize columns."""
    df = pd.read_csv(path)

    # Handle different NYT column names
    date_col = "pub_datetime_utc" if "pub_datetime_utc" in df.columns else "pub_date"
    title_col = "headline" if "headline" in df.columns else "title"
    snippet_col = next(
        (c for c in ["snippet_or_abstract", "snippet", "abstract"] if c in df.columns),
        None
    )
    lead_col = "lead_paragraph" if "lead_paragraph" in df.columns else None

    # Combine snippet + lead_paragraph as body_text (best we can get from NYT)
    snippet = df[snippet_col].fillna("") if snippet_col else ""
    lead = df[lead_col].fillna("") if lead_col else ""
    body = (snippet.astype(str) + " " + lead.astype(str)).str.strip()

    out = pd.DataFrame({
        "pub_date": df[date_col],
        "title": df[title_col],
        "section": df["section"] if "section" in df.columns else "",
        "web_url": df["web_url"],
        "trail_text": snippet,  # use snippet as trail_text equivalent
        "body_text": body,
    })
    return out


dfs = []
for name, info in FILES.items():
    path = info["path"]
    if not os.path.exists(path):
        print(f"  MISSING: {path}")
        continue

    try:
        if info["source"] == "guardian":
            df = load_guardian(path)
        else:
            df = load_nyt(path)

        # Add metadata
        df["source"] = info["source"]
        df["event"] = info["event"]
        df["era"] = info["era"]
        df["origin_file"] = name

        dfs.append(df)
        print(f"  ✓ {name}: {len(df)} rows")
    except Exception as e:
        print(f"  ERROR {name}: {e}")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows before cleaning: {len(df_all)}")

  ✓ guardian_cloudflare_pre: 9 rows
  ✓ guardian_facebook_2021: 139 rows
  ✓ nyt_cloudflare_2019: 1 rows
  MISSING: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/pre_AI/nyt_facebook_outage_2021-10-04.csv
  ✓ nyt_facebook_2021_month: 85 rows
  ✓ guardian_cloudflare_2025: 2 rows
  ✓ guardian_crowdstrike_2024: 64 rows
  ✓ guardian_aws_2025: 35 rows
  ✓ nyt_cloudflare_2025: 1 rows
  ✓ nyt_crowdstrike_2024: 40 rows
  ✓ nyt_aws_2025: 16 rows

Total rows before cleaning: 392


In [5]:
print("=== Cleaning ===")

# 1. Standardize dates
df_all["pub_date"] = pd.to_datetime(df_all["pub_date"], utc=True, errors="coerce")

# 2. Drop duplicates by URL (same article from different queries)
before = len(df_all)
df_all = df_all.drop_duplicates(subset="web_url", keep="first")
print(f"Dedup by URL: {before} -> {len(df_all)} (removed {before - len(df_all)})")

# 3. Drop rows with no title
before = len(df_all)
df_all = df_all.dropna(subset=["title"])
df_all = df_all[df_all["title"].str.strip() != ""]
print(f"Drop empty titles: {before} -> {len(df_all)}")

# 4. Create combined text field for analysis
#    Guardian: use body_text (full text)
#    NYT: use body_text (snippet + lead_paragraph)
df_all["text_for_analysis"] = df_all["body_text"].fillna("").astype(str)
df_all["text_length"] = df_all["text_for_analysis"].str.len()
df_all["has_full_text"] = df_all["text_length"] > 200  # rough threshold

# 5. Flag likely irrelevant articles
#    (articles that don't mention outage-related terms at all)
outage_keywords = [
    "outage", "down", "disruption", "failure", "crash", "error",
    "offline", "interrupt", "glitch", "broke", "unavailable",
    "blue screen", "grounded", "cancelled",
]

def is_relevant(row):
    """Check if article is likely about an outage."""
    text = f"{row['title']} {row['text_for_analysis']}".lower()
    return any(kw in text for kw in outage_keywords)

df_all["likely_relevant"] = df_all.apply(is_relevant, axis=1)
irrelevant = (~df_all["likely_relevant"]).sum()
print(f"Flagged as possibly irrelevant: {irrelevant}")
print("  (NOT auto-removed — review these manually)")

# Show irrelevant articles for manual review
if irrelevant > 0:
    print("\n  Sample irrelevant titles:")
    sample = df_all[~df_all["likely_relevant"]]["title"].head(10)
    for t in sample:
        print(f"    - {t[:80]}")



=== Cleaning ===
Dedup by URL: 392 -> 387 (removed 5)
Drop empty titles: 387 -> 387
Flagged as possibly irrelevant: 132
  (NOT auto-removed — review these manually)

  Sample irrelevant titles:
    - Met officers investigated over Couzens WhatsApp group are still on duty
    - Evidence of ‘vulgar and sexist’ WhatsApp texts ignored, says ex-Met detective
    - How we met: ‘I sent him a Facebook message by accident’
    - How losing a friend to misinformation drove Facebook whistleblower 
    - Facebook putting profit before public good, says whistleblower Frances Haugen
    - ‘Congress will be taking action’: key takeaways from the Facebook whistleblower 
    - First Thing: Facebook ‘harms children, stokes division, weakens democracy’
    - ‘I might delete it’: Facebook’s problem with younger users
    - Facebook whistleblower testimony should prompt new oversight – Schiff
    - Supreme court, Facebook, Fed: three horsemen of democracy’s apocalypse | Robert 


In [6]:
print("=" * 60)
print("MERGED DATASET SUMMARY")
print("=" * 60)

print(f"\nTotal articles: {len(df_all)}")
print(f"Likely relevant: {df_all['likely_relevant'].sum()}")
print(f"Has full text (>200 chars): {df_all['has_full_text'].sum()}")

print("\n--- By Era ---")
print(df_all.groupby("era").agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    likely_relevant=("likely_relevant", "sum"),
))

print("\n--- By Event ---")
print(df_all.groupby(["era", "event"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    likely_relevant=("likely_relevant", "sum"),
))

print("\n--- By Source ---")
print(df_all.groupby(["source"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    avg_text_len=("text_length", "mean"),
))

print("\n--- By Era x Source ---")
print(df_all.groupby(["era", "source"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
))

MERGED DATASET SUMMARY

Total articles: 387
Likely relevant: 255
Has full text (>200 chars): 315

--- By Era ---
         articles  has_full_text  likely_relevant
era                                              
post_ai       153            101              110
pre_ai        234            214              145

--- By Event ---
                          articles  has_full_text  likely_relevant
era     event                                                     
post_ai aws_2025                48             33               31
        cloudflare_2025          1              0                1
        crowdstrike_2024       104             68               78
pre_ai  cloudflare_2019         10             10               10
        facebook_2021          224            204              135

--- By Source ---
          articles  has_full_text  avg_text_len
source                                         
guardian       245            245   5668.473469
nyt            142             70    

In [7]:
# Full merged dataset
df_all.to_csv(f"{BASE}/merged_all_articles.csv", index=False)
print(f"Saved: {BASE}/merged_all_articles.csv ({len(df_all)} rows)")

# Only relevant articles
df_relevant = df_all[df_all["likely_relevant"]].copy()
df_relevant.to_csv(f"{BASE}/merged_relevant_articles.csv", index=False)
print(f"Saved: {BASE}/merged_relevant_articles.csv ({len(df_relevant)} rows)")

# Guardian-only (full text, for primary analysis)
df_guardian = df_relevant[df_relevant["source"] == "guardian"].copy()
df_guardian.to_csv(f"{BASE}/merged_guardian_fulltext.csv", index=False)
print(f"Saved: {BASE}/merged_guardian_fulltext.csv ({len(df_guardian)} rows)")

# Separate postmortem (don't mix with news)
print(f"\nNote: postmortem_all.csv kept separate — use as technical ground truth")

Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_all_articles.csv (387 rows)
Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_relevant_articles.csv (255 rows)
Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_guardian_fulltext.csv (203 rows)

Note: postmortem_all.csv kept separate — use as technical ground truth


In [8]:
print("=== Sample Articles Per Event ===\n")
for event in df_all["event"].unique():
    subset = df_all[df_all["event"] == event]
    print(f"\n--- {event} ({len(subset)} articles) ---")
    sample = subset.head(3)
    for _, row in sample.iterrows():
        print(f"  [{row['source']}] {row['title'][:70]}")
        print(f"    text_length={row['text_length']}, relevant={row['likely_relevant']}")

=== Sample Articles Per Event ===


--- cloudflare_2019 (10 articles) ---
  [guardian] CloudFlare on censorship: 'A website is speech. It is not a bomb'
    text_length=2893, relevant=True
  [guardian] Web services firm CloudFlare accused by Anonymous of helping Isis
    text_length=2655, relevant=True
  [guardian] Web giant Cloudflare stores extreme neo-Nazi content on UK soil
    text_length=4380, relevant=True

--- facebook_2021 (224 articles) ---
  [guardian] Jessica Rowe podcast turns sour over Pauline Hanson interview on ‘why 
    text_length=6981, relevant=True
  [guardian] Don’t say ‘privilege’: can the left find better words for talking with
    text_length=14860, relevant=True
  [guardian] Boris Johnson doesn’t fear Labour. His biggest problem will be his own
    text_length=5486, relevant=True

--- crowdstrike_2024 (104 articles) ---
  [guardian] The US government is right to investigate Nvidia for alleged unfair pr
    text_length=5376, relevant=True
  [guardian] Tasmania f